# BusyBox feature entropy evaluation

This notebook joins the feature-cardinality stage with the feature-model-cardinality stage, computes binary entropy for each feature, and plots the distributions for the latest completed BusyBox version plus all available versions over time.


In [ ]:
from decimal import Decimal, getcontext
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    plt = None
    HAS_MATPLOTLIB = False

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    px = None
    HAS_PLOTLY = False

getcontext().prec = 80


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "stages").is_dir() and (path / "experiments").is_dir():
            return path
    raise FileNotFoundError("Could not find repository root containing stages/ and experiments/.")


REPO_ROOT = find_repo_root()
STAGES = REPO_ROOT / "stages"
EXPERIMENT_DIR = REPO_ROOT / "experiments" / "busybox-entropy"

FEATURE_MODEL_CARDINALITY_CSV = STAGES / "12_solve_feature_model_cardinality_sharp_sat" / "output.csv"
FEATURE_CARDINALITY_CSV = STAGES / "13_solve_feature_cardinality_emse_2023_ganak" / "output.csv"
print("Plotly available:", HAS_PLOTLY)
print("Matplotlib available:", HAS_MATPLOTLIB)
FEATURE_MODEL_CARDINALITY_CSV, FEATURE_CARDINALITY_CSV


## Load and normalize stage outputs


In [ ]:
DIMACS_RE = re.compile(r"\[(?P<timestamp>\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})\](?P<revision>[^/]+)\.dimacs$")
QUERY_RE = re.compile(r"^(?P<query_kind>\S+)\s+\+(?P<feature>.+)$")


def parse_dimacs_file(dimacs_file):
    match = DIMACS_RE.search(str(dimacs_file))
    if not match:
        return pd.Series({"revision": pd.NA, "model_timestamp": pd.NaT})
    timestamp = pd.to_datetime(match.group("timestamp"), format="%Y-%m-%d-%H-%M-%S", errors="coerce")
    return pd.Series({"revision": match.group("revision"), "model_timestamp": timestamp})


def version_key(revision):
    if pd.isna(revision):
        return tuple()
    return tuple(int(part) for part in re.findall(r"\d+", str(revision)))


def parse_feature_query(query):
    match = QUERY_RE.match(str(query))
    if match:
        return pd.Series(match.groupdict())
    return pd.Series({"query_kind": pd.NA, "feature": str(query)})


def parse_big_int(value):
    text = str(value).strip()
    return int(text) if re.fullmatch(r"\d+", text) else pd.NA


fm_cardinality = pd.read_csv(FEATURE_MODEL_CARDINALITY_CSV, dtype=str)
feature_cardinality = pd.read_csv(FEATURE_CARDINALITY_CSV, dtype=str)

fm_cardinality = fm_cardinality[fm_cardinality["dimacs_query"].eq("void")].copy()
fm_cardinality = fm_cardinality.join(fm_cardinality["dimacs_file"].apply(parse_dimacs_file))
fm_cardinality["feature_model_cardinality"] = fm_cardinality["sharp_sat"].apply(parse_big_int)
fm_cardinality = fm_cardinality[["dimacs_file", "revision", "model_timestamp", "feature_model_cardinality"]]

feature_cardinality = feature_cardinality.join(feature_cardinality["dimacs_file"].apply(parse_dimacs_file))
feature_cardinality = feature_cardinality.join(feature_cardinality["dimacs_query"].apply(parse_feature_query))
feature_cardinality["feature_cardinality"] = feature_cardinality["sharp_sat"].apply(parse_big_int)
feature_cardinality = feature_cardinality[[
    "dimacs_file",
    "revision",
    "model_timestamp",
    "query_kind",
    "feature",
    "feature_cardinality",
]]

display(fm_cardinality.head())
display(feature_cardinality.head())
print(f"Loaded {len(fm_cardinality):,} feature-model cardinalities and {len(feature_cardinality):,} feature cardinalities.")


## Compute normalized cardinality and entropy

For a feature `f`, `p = c(f) / c(fm)`. The binary entropy is `-p*log2(p) - (1-p)*log2(1-p)` with entropy set to `0` when `p` is exactly `0` or `1`.


In [ ]:
def probability(feature_count, model_count):
    if pd.isna(feature_count) or pd.isna(model_count) or model_count == 0:
        return np.nan
    return float(Decimal(feature_count) / Decimal(model_count))


def binary_entropy(p):
    if pd.isna(p):
        return np.nan
    if p <= 0.0 or p >= 1.0:
        return 0.0
    return -(p * math.log2(p) + (1.0 - p) * math.log2(1.0 - p))


entropy = feature_cardinality.merge(
    fm_cardinality[["dimacs_file", "feature_model_cardinality"]],
    on="dimacs_file",
    how="inner",
)
entropy["normalized_feature_cardinality"] = [
    probability(feature_count, model_count)
    for feature_count, model_count in zip(entropy["feature_cardinality"], entropy["feature_model_cardinality"])
]
entropy["entropy"] = entropy["normalized_feature_cardinality"].apply(binary_entropy)
entropy["version_key"] = entropy["revision"].apply(version_key)

version_order = (
    entropy[["revision", "model_timestamp", "version_key"]]
    .drop_duplicates("revision")
    .sort_values(["model_timestamp", "version_key"], kind="stable")
    .reset_index(drop=True)
)
version_order["version_index"] = np.arange(len(version_order))
entropy = entropy.merge(version_order[["revision", "version_index"]], on="revision", how="left")

entropy = entropy.sort_values(["version_index", "feature"], kind="stable").reset_index(drop=True)

display(entropy.head())
print(f"Computed entropy for {len(entropy):,} feature/version rows across {entropy['revision'].nunique():,} versions.")


## Latest available BusyBox version


In [ ]:
latest_revision = version_order.sort_values(["model_timestamp", "version_key"], kind="stable").iloc[-1]["revision"]
latest = entropy[entropy["revision"].eq(latest_revision)].copy()

print(f"Latest available revision with feature-cardinality rows: {latest_revision}")
print(f"Features available for this revision so far: {len(latest):,}")


def add_rank_axis(df, sort_column):
    ranked = df.sort_values(sort_column, ascending=False, kind="stable").reset_index(drop=True).copy()
    ranked["feature_rank"] = np.linspace(0.0, 1.0, len(ranked)) if len(ranked) > 1 else 0.0
    return ranked


latest_entropy = add_rank_axis(latest, "entropy")
if HAS_PLOTLY:
    fig = px.scatter(
        latest_entropy,
        x="feature_rank",
        y="entropy",
        hover_data=["revision", "feature", "normalized_feature_cardinality", "feature_cardinality", "feature_model_cardinality"],
        labels={"feature_rank": "Features ordered by descending entropy", "entropy": "Entropy"},
        title=f"BusyBox {latest_revision}: feature entropy distribution",
    )
    fig.update_traces(marker={"size": 5, "opacity": 0.75})
    fig.update_layout(xaxis_range=[0, 1])
    fig.show()
elif HAS_MATPLOTLIB:
    ax = latest_entropy.plot.scatter(x="feature_rank", y="entropy", s=16, alpha=0.75, figsize=(9, 4))
    ax.set_xlim(0, 1)
    ax.set_title(f"BusyBox {latest_revision}: feature entropy distribution")
    ax.set_xlabel("Features ordered by descending entropy")
    ax.set_ylabel("Entropy")
    plt.show()
else:
    print("Install plotly for interactive plots or matplotlib for static plots.")

latest_cardinality = add_rank_axis(latest, "normalized_feature_cardinality")
if HAS_PLOTLY:
    fig = px.scatter(
        latest_cardinality,
        x="feature_rank",
        y="normalized_feature_cardinality",
        hover_data=["revision", "feature", "entropy", "feature_cardinality", "feature_model_cardinality"],
        labels={
            "feature_rank": "Features ordered by descending normalized cardinality",
            "normalized_feature_cardinality": "c(f) / c(fm)",
        },
        title=f"BusyBox {latest_revision}: normalized feature-cardinality distribution",
    )
    fig.update_traces(marker={"size": 5, "opacity": 0.75})
    fig.update_layout(xaxis_range=[0, 1], yaxis_range=[0, 1])
    fig.show()
elif HAS_MATPLOTLIB:
    ax = latest_cardinality.plot.scatter(x="feature_rank", y="normalized_feature_cardinality", s=16, alpha=0.75, figsize=(9, 4))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(f"BusyBox {latest_revision}: normalized feature-cardinality distribution")
    ax.set_xlabel("Features ordered by descending normalized cardinality")
    ax.set_ylabel("c(f) / c(fm)")
    plt.show()
else:
    print("Install plotly for interactive plots or matplotlib for static plots.")


## Interactive distributions over time

Each version is sorted independently by the plotted metric. The x-axis is the normalized feature rank within that version, the y-axis is the version order, and the z-axis is the metric value.


In [ ]:
def ranked_over_time(df, sort_column):
    ranked_frames = []
    for _, group in df.groupby("revision", sort=False):
        ranked = add_rank_axis(group, sort_column)
        ranked_frames.append(ranked)
    return pd.concat(ranked_frames, ignore_index=True) if ranked_frames else pd.DataFrame()


def add_sparse_version_ticks(fig):
    if version_order.empty:
        return fig
    step = max(1, math.ceil(len(version_order) / 12))
    ticks = version_order.iloc[::step]
    if version_order.iloc[-1]["revision"] not in set(ticks["revision"]):
        ticks = pd.concat([ticks, version_order.tail(1)])
    fig.update_layout(scene={"yaxis": {"tickmode": "array", "tickvals": ticks["version_index"], "ticktext": ticks["revision"]}})
    return fig


entropy_time = ranked_over_time(entropy, "entropy")
if HAS_PLOTLY:
    fig = px.scatter_3d(
        entropy_time,
        x="feature_rank",
        y="version_index",
        z="entropy",
        color="version_index",
        hover_data=["revision", "feature", "normalized_feature_cardinality"],
        labels={"feature_rank": "Feature rank", "version_index": "BusyBox version", "entropy": "Entropy"},
        title="BusyBox feature entropy distributions over time",
    )
    fig.update_traces(marker={"size": 3, "opacity": 0.7})
    fig.update_layout(scene={"xaxis": {"range": [0, 1]}, "zaxis": {"range": [0, 1]}})
    add_sparse_version_ticks(fig)
    fig.show()
elif HAS_MATPLOTLIB:
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(entropy_time["feature_rank"], entropy_time["version_index"], entropy_time["entropy"], s=5, alpha=0.6)
    ax.set_xlim(0, 1)
    ax.set_zlim(0, 1)
    ax.set_xlabel("Feature rank")
    ax.set_ylabel("BusyBox version")
    ax.set_zlabel("Entropy")
    ax.set_title("BusyBox feature entropy distributions over time")
    plt.show()
else:
    print("Install plotly for interactive 3D plots or matplotlib for static 3D plots.")

cardinality_time = ranked_over_time(entropy, "normalized_feature_cardinality")
if HAS_PLOTLY:
    fig = px.scatter_3d(
        cardinality_time,
        x="feature_rank",
        y="version_index",
        z="normalized_feature_cardinality",
        color="version_index",
        hover_data=["revision", "feature", "entropy"],
        labels={"feature_rank": "Feature rank", "version_index": "BusyBox version", "normalized_feature_cardinality": "c(f) / c(fm)"},
        title="BusyBox normalized feature-cardinality distributions over time",
    )
    fig.update_traces(marker={"size": 3, "opacity": 0.7})
    fig.update_layout(scene={"xaxis": {"range": [0, 1]}, "zaxis": {"range": [0, 1]}})
    add_sparse_version_ticks(fig)
    fig.show()
elif HAS_MATPLOTLIB:
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(cardinality_time["feature_rank"], cardinality_time["version_index"], cardinality_time["normalized_feature_cardinality"], s=5, alpha=0.6)
    ax.set_xlim(0, 1)
    ax.set_zlim(0, 1)
    ax.set_xlabel("Feature rank")
    ax.set_ylabel("BusyBox version")
    ax.set_zlabel("c(f) / c(fm)")
    ax.set_title("BusyBox normalized feature-cardinality distributions over time")
    plt.show()
else:
    print("Install plotly for interactive 3D plots or matplotlib for static 3D plots.")


## Current experiment coverage


In [ ]:
coverage = (
    entropy.groupby(["version_index", "revision"], as_index=False)
    .agg(
        available_features=("feature", "nunique"),
        mean_entropy=("entropy", "mean"),
        max_entropy=("entropy", "max"),
        mean_normalized_cardinality=("normalized_feature_cardinality", "mean"),
    )
    .sort_values("version_index", kind="stable")
)

display(coverage.tail(20))
